# Brique 0 — Comprendre Python avant d'écrire une ligne de code

Tu connais déjà Java (tes TD de 1A et 2A). L'objectif de ce notebook n'est pas de t'apprendre la syntaxe Python
(ça viendra dans la Brique 1), mais de comprendre **ce qu'il y a sous le capot** et **pourquoi** Python ressemble
et ne ressemble pas à Java. Sans ça, tu codes en Python avec une tête de développeur Java, et certaines choses
(le typage, la lenteur relative, le GIL, les conventions de projet) restent mystérieuses.

Plan :
1. Histoire et philosophie
2. Interprété vs compilé — ce que devient vraiment ton code
3. CPython et les autres implémentations
4. Typage dynamique, duck typing — la vraie différence avec Java
5. Le Global Interpreter Lock (GIL)
6. Gestion de la mémoire
7. Bibliographie

## 1. Histoire et philosophie

Python est créé par **Guido van Rossum**, chercheur néerlandais au CWI (Amsterdam), à partir de décembre 1989,
publié en 1991. Le nom vient des Monty Python (l'émission comique), pas du serpent — Guido voulait un nom
"court, unique, et légèrement mystérieux".

Avant Python, Guido travaillait sur un langage pédagogique appelé **ABC**. ABC avait de bonnes idées
(indentation significative, lisibilité) mais était trop fermé pour être étendu facilement. Python reprend ces
idées et les rend extensibles : on peut écrire des modules, importer des bibliothèques C, etc.

**Les dates qui comptent :**

| Version | Année | Pourquoi ça compte |
|---|---|---|
| Python 1.0 | 1994 | Premier langage "complet" |
| Python 2.0 | 2000 | Garbage collector moderne, list comprehensions |
| Python 3.0 | 2008 | Rupture volontaire de compatibilité (`print` devient une fonction, strings Unicode par défaut, division `/` toujours flottante) |
| Fin de vie Python 2 | 1er janvier 2020 | Si tu vois encore du code `print "..."` sans parenthèses, c'est du Python 2 mort |

**La philosophie** est écrite dans un easter egg officiel. Tape ceci dans un interpréteur Python :

```python
import this
```

Ça affiche le "Zen of Python", 19 principes. Les plus importants à retenir pour la suite de ce parcours :

- *Explicit is better than implicit* — pas de magie cachée si on peut l'éviter.
- *Simple is better than complex* — on reviendra souvent là-dessus pendant la Brique POO (savoir **ne pas**
  sur-architecturer).
- *Readability counts* — Python est conçu pour être lu, pas seulement exécuté.
- *There should be one — and preferably only one — obvious way to do it* — contraste fort avec Perl ou C++
  qui offrent souvent dix façons de faire la même chose.

In [ ]:
import this

## 2. Interprété vs compilé — ce que devient vraiment ton code

**En Java**, tu compiles ton `.java` en **bytecode JVM** (`.class`), puis la JVM exécute ce bytecode, avec un
JIT (Just-In-Time compiler) qui traduit les parties chaudes en code machine natif à l'exécution pour aller plus
vite. C'est pour ça que Java est "compilé puis exécuté par une machine virtuelle".

**En Python (CPython, l'implémentation que tu utilises par défaut)**, le principe est étonnamment proche :

1. Ton fichier `.py` est **compilé** en bytecode Python (pas du code machine — un bytecode propre à Python,
   que tu peux voir avec le module `dis`).
2. Ce bytecode est mis en cache dans un fichier `.pyc` (dans un dossier `__pycache__/`), pour ne pas recompiler
   si le fichier source n'a pas changé.
3. La **machine virtuelle CPython** (écrite en C) exécute ce bytecode, instruction par instruction.

La vraie différence avec Java n'est donc pas "interprété vs compilé" (Python compile aussi, vers un bytecode) :
c'est que **CPython n'a historiquement pas de JIT** (pas de passage au code machine natif), alors que la JVM en a
un depuis longtemps. C'est l'une des raisons pour lesquelles du code Python pur est souvent plus lent que du Java
équivalent sur du calcul intensif — et pourquoi NumPy/Pandas existent (ils délèguent le calcul à du C compilé,
voir la Brique 4).

In [ ]:
import dis

def addition(a, b):
    return a + b

# Ceci montre le bytecode réellement exécuté par la machine virtuelle CPython.
# Compare avec le "javap -c" que tu aurais utilisé en Java pour voir le bytecode JVM d'une classe.
dis.dis(addition)

**Exercice de lecture** (pas de code à écrire, juste à observer) : relance la cellule au-dessus en changeant
`a + b` en `a * b`, puis en `a if a > b else b`. Regarde comment le bytecode change. Tu n'as pas besoin de
comprendre chaque instruction (`LOAD_FAST`, `BINARY_ADD`, ...) — retiens juste que **chaque ligne Python devient
une suite d'instructions bas niveau**, exactement comme en Java, juste avec un format différent (bytecode Python
au lieu de bytecode JVM).

## 3. CPython et les autres implémentations

"Python" est un **langage** (une spécification). "CPython" est **l'implémentation de référence**, écrite en C
— c'est celle que tu utilises à 99% du temps, celle que tu lances avec `python` dans un terminal.

Il existe d'autres implémentations, chacune avec un but précis :

| Implémentation | Écrite en | Pourquoi elle existe |
|---|---|---|
| **CPython** | C | La référence, celle que tout le monde utilise par défaut |
| **PyPy** | Python (auto-hébergé) | A un vrai JIT → souvent 4 à 10× plus rapide que CPython sur du code pur Python |
| **Jython** | Java | Compile vers du bytecode JVM → interopère avec des bibliothèques Java |
| **IronPython** | C# | Pareil, mais pour l'écosystème .NET |
| **MicroPython** | C | Version allégée pour microcontrôleurs (Raspberry Pi Pico, etc.) |

Pourquoi c'est utile de le savoir : si un jour tu entends "on a basculé ce service sur PyPy pour la perf", tu
sais que ça veut dire "même langage, VM différente, avec JIT". Et si tu lis `sys.implementation.name` dans du
code et que ça affiche `cpython`, tu sais maintenant ce que ça signifie.

In [ ]:
import sys
print(sys.implementation.name)
print(sys.version)

## 4. Typage dynamique et duck typing — la vraie différence avec Java

En Java, chaque variable a un **type déclaré à la compilation** : `int x = 5;` — le compilateur vérifie les
types avant même que le programme tourne. C'est du **typage statique**.

En Python, une variable est juste un nom qui **pointe vers un objet**, et c'est l'objet qui a un type, pas le
nom. Le type n'est vérifié qu'à l'exécution. C'est le **typage dynamique**.

In [ ]:
x = 5        # x pointe vers un int
print(type(x))
x = "cinq"    # x pointe maintenant vers un str. Interdit en Java sans recast, normal en Python.
print(type(x))

Conséquence directe : le **duck typing** ("si ça marche comme un canard, c'est un canard"). En Java, pour que
deux classes soient interchangeables dans une fonction, elles doivent partager une interface ou une classe mère
déclarée explicitement. En Python, il suffit que l'objet **ait la bonne méthode** — aucun lien de parenté requis.

In [ ]:
class Canard:
    def parler(self):
        return "Coin coin"

class Humain:
    def parler(self):
        return "Je imite un canard : coin coin"

# Ces deux classes n'ont AUCUN lien entre elles (pas d'interface commune, pas d'héritage).
# Pourtant cette fonction fonctionne avec les deux : elle ne demande pas "es-tu un Canard ?",
# elle demande juste "as-tu une méthode parler() ?".
def faire_parler(quelque_chose):
    print(quelque_chose.parler())

faire_parler(Canard())
faire_parler(Humain())

C'est une arme à double tranchant, et on en reparlera beaucoup en Brique 2 (POO) : ça rend Python très souple,
mais ça déplace la détection des erreurs de type de la compilation vers l'exécution (parfois en production).
C'est pour ça que les *type hints* (`def f(x: int) -> str:`) et des outils comme `mypy` existent : ils ramènent
une partie de la sécurité statique de Java, en option, sans changer le fonctionnement réel du langage.

## 5. Le Global Interpreter Lock (GIL)

C'est la particularité la plus citée, et souvent mal expliquée, de CPython.

**Le problème que le GIL résout** : CPython gère la mémoire par comptage de références (section suivante). Si
deux threads modifiaient ce compteur en même temps sans protection, on aurait des *race conditions* internes à
l'interpréteur lui-même (pas juste dans ton code).

**La solution choisie** (dans les années 90, et jamais retirée depuis pour des raisons de compatibilité) : un
verrou global unique. **À un instant donné, un seul thread exécute du bytecode Python**, même sur une machine à
16 cœurs.

**Ce que ça veut dire concrètement :**

| Situation | Le GIL bloque-t-il la parallélisation ? |
|---|---|
| Threads qui font du calcul Python pur (boucles, calculs) | Oui — aucun gain de vitesse, parfois une perte |
| Threads qui attendent un réseau, un disque, une base de données (I/O) | Non — le GIL est relâché pendant l'attente, donc le threading reste utile pour l'I/O concurrent |
| Code NumPy/Pandas qui appelle du C en dessous | Souvent non — ces bibliothèques relâchent le GIL pendant leurs calculs C |
| Besoin de vrai parallélisme CPU | On utilise le module `multiprocessing` (plusieurs processus, chacun avec son propre interpréteur et donc son propre GIL) plutôt que `threading` |

Retiens la règle pratique : **threading pour l'I/O, multiprocessing pour le calcul CPU**. On y reviendra quand
ce sera utile dans les briques suivantes.

*Note pour plus tard : un projet appelé "PEP 703" vise à rendre le GIL optionnel dans les futures versions de
Python. Tu n'as pas besoin de ça maintenant, mais si tu entends parler de "free-threaded Python", c'est de ça
qu'il s'agit.*

## 6. Gestion de la mémoire

En Java, la JVM a un garbage collector qui scanne périodiquement le tas pour retrouver les objets inaccessibles.

En CPython, le mécanisme principal est différent : le **comptage de références**. Chaque objet garde un compteur
du nombre de noms qui pointent vers lui. Dès que ce compteur tombe à zéro, l'objet est détruit **immédiatement**
(pas besoin d'attendre un passage du garbage collector).

In [ ]:
import sys

a = [1, 2, 3]
print(sys.getrefcount(a))  # au moins 2 : la variable 'a', + la référence temporaire créée par getrefcount lui-même

b = a  # un deuxième nom pointe vers le même objet liste
print(sys.getrefcount(a))  # le compteur a augmenté

del b  # on retire ce nom
print(sys.getrefcount(a))  # revenu à la valeur de départ

Le comptage de références seul ne suffit pas : il ne détecte pas les **cycles** (deux objets qui se référencent
mutuellement, donc dont le compteur ne tombe jamais naturellement à zéro, même s'ils sont inaccessibles depuis le
reste du programme). C'est pour ça que CPython a *en plus* un vrai garbage collector (module `gc`), qui ne scanne
que périodiquement, uniquement pour casser ces cycles. Le comptage de références fait le gros du travail en
continu ; le GC générationnel nettoie les cas particuliers.

## 7. Bibliographie et pour aller plus loin

- *Zen of Python* (`import this`) — les 19 principes de design.
- **PEP 8** — le guide de style officiel du code Python (indentation, nommage...). On l'appliquera dès la
  Brique 1.
- **PEP 20** — le texte formel du Zen of Python.
- Documentation officielle, *Design and History FAQ* :
  https://docs.python.org/3/faq/design.html — l'historique écrit par les créateurs eux-mêmes.
- Pour creuser le bytecode et la VM : module `dis` (déjà vu plus haut), et le livre *CPython Internals* de
  Anthony Shaw si tu veux vraiment descendre dans les détails d'implémentation un jour.
- Pour creuser le GIL : l'article historique de David Beazley, *Understanding the Python GIL* (talk PyCon), très
  pédagogique en vidéo.

---

**Prochaine étape** : `01_pratique.ipynb` — comment structurer un repo Python comme ceux que tu vois chez
Cozycozy (src/, lib/, requirements.txt, .env, .gitignore, etc.), maintenant que tu sais *pourquoi* ces outils
existent (environnements virtuels pour isoler les dépendances d'un interpréteur qui n'a pas de typage statique
pour te protéger des incompatibilités de version, etc.).